**Author:** Christopher Millward<br>
**Date:** June 29, 2026

The purpose of this notebook is to just verify that the axes in the R matrices have been defined as expected (according to ISB). The humerus orientations are described wrt the torso sensor. From the diagram Dan drew (`/resources/sensor_axis_diagram.pdf`), we know the IMU axes were in the following directions:

| ISB | Toroso IMU | Right IMU | Left IMU |
| --- | ---------- | --------- | -------- |
| X   | Z          | -X        | X        |
| Y   | -Y         | -Y        | -Y       |
| Z   | X          | Z         | Z        |

Which means for each side, we have to perform the following corrections:

| Axis | Right IMU | Left IMU |
| ---- | --------- | -------- |
| ---  | --------  | -------- | 

**FILL IN THIS TABLE AND MAKE THE ADJUSTMENTS IN THE NOTEBOOK CODE ACCORDINGLY. THIS NOTEBOOK IS INCOMPLETE**

I will open a single file (assuming they're all defined accordingly) and plot the R matrix axis triads in an animated time-series. I will fix the viewport to the ISB coordinate system and verify that all axes are in the expected direction.


In [ ]:
import os 
import sys

# set root folder to project root
root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
# CRITICAL: This tells Jupyter to use an interactive window that allows real-time loop updates.
# If '%matplotlib notebook' fails or is not installed, change it to '%matplotlib widget'
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from utils.data_loading import load_participant_details
from utils.kinematics.general_helpers import create_rotation_matrices

participant_details = load_participant_details('../data/raw_normalized_data/participant_details.xlsx')
path = participant_details[0]['filename']
data = np.loadtxt(f'../data/raw_normalized_data/{path}', delimiter='\t', skiprows=1, usecols=range(1,19))

In [ ]:
# Create rotation matrices
R_left = create_rotation_matrices(data, 'left')
R_right = create_rotation_matrices(data, 'right')

In [ ]:
# Set up the figure with 1 row, 2 columns of 3D subplots
fig, axes = plt.subplots(1, 2, subplot_kw={'projection': '3d'}, figsize=(10, 6))
plt.suptitle('IMU Axes Verification (Shoulder Motion)', fontsize=14, fontweight='bold')

# Axis styling and labels
titles = ['Left Shoulder', 'Right Shoulder']
data_sources = [R_left, R_right]
axes_elements = []

for idx, ax in enumerate(axes):
    ax.set_title(titles[idx], fontsize=12)
    
    # Set uniform 3D limits
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    
    # Label the global/laboratory coordinate system
    ax.set_xlabel('Global X')
    ax.set_ylabel('Global Y')
    ax.set_zlabel('Global Z')

    # Rotate to easily match ISB conventions for humerothoracic.
    if idx == 0:  # Left Shoulder subplot
        # Elev=0 places Y up. Azim=-180 points X forward, Z to the left
        ax.view_init(elev=0, azim=0, roll=90)
    else:         # Right Shoulder subplot
        # (elev=180, azim=0, roll=90) is ISB view
        ax.view_init(elev=180, azim=0, roll=90)
    
    # Initialize 3 lines for the triad: Red (X), Green (Y), Blue (Z)
    line_x, = ax.plot([], [], [], color='red', linewidth=3, label='Local X')
    line_y, = ax.plot([], [], [], color='green', linewidth=3, label='Local Y')
    line_z, = ax.plot([], [], [], color='blue', linewidth=3, label='Local Z')
    line_hum, = ax.plot([], [], [], color='black', linewidth=3, label='Humerus')
    
    # Draw a small static gray dot at the joint origin (0,0,0)
    ax.scatter([0], [0], [0], color='gray', s=20)
    
    if idx == 0:
        ax.legend(loc='upper left')
        
    # Store line references to update them frame-by-frame
    axes_elements.append((line_x, line_y, line_z, line_hum))

def update(frame):
    """Updates the 3D lines for both subplots at the current frame index."""
    for idx, (line_x, line_y, line_z, line_hum) in enumerate(axes_elements):
        # Extract the current 3x3 rotation matrix
        R = data_sources[idx][frame]
        
        # Columns of R are the local X, Y, Z unit vectors
        v_x = R[:, 0]
        v_y = R[:, 1]
        v_z = R[:, 2]
        v_hum = v_y *-1
        
        # Update line segments drawing from origin (0,0,0) to the vector tip
        line_x.set_data_3d([0, v_x[0]], [0, v_x[1]], [0, v_x[2]])
        line_y.set_data_3d([0, v_y[0]], [0, v_y[1]], [0, v_y[2]])
        line_z.set_data_3d([0, v_z[0]], [0, v_z[1]], [0, v_z[2]])
        line_hum.set_data_3d([0, v_hum[0]], [0, v_hum[1]], [0, v_hum[2]])
    return [line for elements in axes_elements for line in elements]

# Create the animation loop
# frames=4000 loops through all matrices. interval=25 runs at ~40 FPS.
# blit=True ensures smooth rendering by only updating altered pixels.
ani = FuncAnimation(
    fig, 
    update, 
    # frames=round(len(R_left)/10),
    frames=range(1, 100),
    interval=5, 
    blit=True
)
update(0)  # Call update once to initialize the lines
plt.tight_layout()
plt.show()